# RT-IoT2022 - Data Preprocessing

In this notebook we will see the preprocessing phase of our dataset for anomaly detection. The main steps are:

- load and clean the dataset
- separate normal and anomalous traffic
- create training, validation and test sets
- encode categorical features
- scale numerical features
- save the processed datasets

The training set has only the normal traffic while validation and test sets have both normal and anomalous traffic.

In [35]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
df = pd.read_csv("../data/raw/RT_IOT2022.csv")
print("Dataset shape:", df.shape)

df = df.drop(columns=["Unnamed: 0"])
print("Dataset shape after removing index column:", df.shape)

In [ ]:
normal_classes = ["MQTT_Publish", "Thing_Speak", "Wipro_bulb"]

df["Label"] = df["Attack_type"].apply(
    lambda x: "Normal" if x in normal_classes else "Anomaly"
)

print(df["Label"].value_counts())

In [ ]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicated rows:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicated rows after removal:", df.duplicated().sum())
print("Dataset shape after removing duplicates:", df.shape)

In [ ]:
normal_df = df[df["Label"] == "Normal"].copy()
anomaly_df = df[df["Label"] == "Anomaly"].copy()

print("Normal samples:", len(normal_df))
print("Anomaly samples:", len(anomaly_df))

In [ ]:
normal_train, normal_temp = train_test_split(
    normal_df,
    test_size = 0.40,
    random_state=42,
    stratify=normal_df["Attack_type"]
)

normal_val, normal_test = train_test_split(
    normal_temp,
    test_size=0.50,
    random_state=42,
    stratify=normal_temp["Attack_type"]
)

print("Normal training samples:", len(normal_train))
print("Normal validation samples:", len(normal_val))
print("Normal test samples:", len(normal_test))

In [ ]:
anomaly_val, anomaly_test = train_test_split(
    anomaly_df,
    test_size = 0.50,
    random_state = 42,
    stratify = anomaly_df["Attack_type"]
)

print("Anomalies in validation:", len(anomaly_val))
print("Anomalies in test:", len(anomaly_test))

In [ ]:
val_df = pd.concat([normal_val, anomaly_val])
test_df = pd.concat([normal_test, anomaly_test])

val_df = val_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

normal_train = normal_train.reset_index(drop=True)

print("Training set:", normal_train.shape)
print("Validation set:", val_df.shape)
print("Test set:", test_df.shape)

print("\nValidation labels:")
print(val_df["Label"].value_counts())

print("\nTest labels:")
print(test_df["Label"].value_counts())

In [ ]:
feature_columns = [
    column for column in df.columns
    if column not in ["Attack_type", "Label"]
]

X_train = normal_train[feature_columns].copy()
X_val = val_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

train_metadata = normal_train[["Attack_type", "Label"]].copy()
val_metadata = val_df[["Attack_type", "Label"]].copy()
test_metadata = test_df[["Attack_type", "Label"]].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

In [ ]:
numeric_columns = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_columns))
print("Categorical features:", len(categorical_columns))

print("\nCategorical features:")
print(categorical_columns)

In [ ]:
print(
    "Infinite values in training:",
    np.isinf(X_train[numeric_columns]).sum().sum()
)

print(
    "Infinite values in validation:",
    np.isinf(X_val[numeric_columns]).sum().sum()
)

print(
    "Infinite values in test:",
    np.isinf(X_test[numeric_columns]).sum().sum()
)

In [ ]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoder.fit(X_train[categorical_columns])
X_train_cat = encoder.transform(X_train[categorical_columns])
X_val_cat = encoder.transform(X_val[categorical_columns])
X_test_cat = encoder.transform(X_test[categorical_columns])

categorical_feature_names = encoder.get_feature_names_out(
    categorical_columns
)

print(categorical_feature_names)

In [47]:
scaler = StandardScaler()
scaler.fit(X_train[numeric_columns])
X_train_num = scaler.transform(X_train[numeric_columns])
X_val_num = scaler.transform(X_val[numeric_columns])
X_test_num = scaler.transform(X_test[numeric_columns])

In [ ]:
final_columns = numeric_columns +list(categorical_feature_names)

X_train_processed = pd.DataFrame(
    np.hstack([X_train_num, X_train_cat]),
    columns = final_columns
)

X_val_processed = pd.DataFrame(
    np.hstack([X_val_num, X_val_cat]),
    columns = final_columns
)

X_test_processed = pd.DataFrame(
    np.hstack([X_test_num, X_test_cat]),
    columns = final_columns
)

print("Processed training set:", X_train_processed.shape)
print("Processed validation set:", X_val_processed.shape)
print("Processed test set:", X_test_processed.shape)

In [ ]:
print(
    "Missing values in training:",
    X_train_processed.isnull().sum().sum()
)

print(
    "Missing values in validation:",
    X_val_processed.isnull().sum().sum()
)

print(
    "Missing values in test:",
    X_test_processed.isnull().sum().sum() 
)

print(
    "Infinite values in processed training:",
    np.isinf(X_train_processed.to_numpy()).sum()
)

print(
    "Infinite values in processed validation:",
    np.isinf(X_val_processed.to_numpy()).sum()
)

print(
   "Infinite values in processed test:",
   np.isinf(X_test_processed.to_numpy()).sum()
)

In [50]:
X_train_processed.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_val_processed.to_csv(
    "../data/processed/X_val.csv",
    index=False
)

X_test_processed.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

train_metadata.to_csv(
    "../data/processed/train_metadata.csv",
    index=False
)

val_metadata.to_csv(
    "../data/processed/val_metadata.csv",
    index=False
)

test_metadata.to_csv(
    "../data/processed/test_metadata.csv",
    index=False
)

In [ ]:
print("Preprocessing completed successfully")

print("Training:")
print(X_train_processed.shape)

print("\nvalidation:")
print(X_val_processed.shape)

print("\nTest:")
print(X_test_processed.shape)

The dataset, as we can see, was divided into training, validation and testing. The first one contains only normal IoT traffic while the other ones contain both normal and anomalous IoT traffic.

Thanks to this code, in the end we'll obtain processed datasets used by the anomaly detection models